# Meridian Trade & Logistics — Exploratory Data Analysis

**Client:** Meridian Trade & Logistics Pte. Ltd. (Singapore)  
**Analyst:** Nurbol Sultanov  
**Date:** June 2023  

Initial exploration of shipment data across Southeast Asian and East Asian routes (2022–2024).  
Focus: delivery delays, route performance, and seasonal patterns.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [2]:
shipments = pd.read_csv('../data/raw/shipments.csv', parse_dates=[
    'booking_date', 'scheduled_departure', 'actual_departure',
    'scheduled_arrival', 'actual_arrival'
])
ports = pd.read_csv('../data/raw/ports.csv')

print(f'Shipments: {shipments.shape}')
print(f'Ports: {ports.shape}')

Shipments: (8000, 20)
Ports: (15, 4)


In [3]:
print(f'Date range: {shipments["booking_date"].min().date()} to {shipments["booking_date"].max().date()}')
print(f'Unique origins: {shipments["origin_port"].nunique()}')
print(f'Unique destinations: {shipments["destination_port"].nunique()}')
mode_counts = shipments['transport_mode'].value_counts()
print(f'Transport modes: {mode_counts.index[0]} {mode_counts.values[0]}, {mode_counts.index[1]} {mode_counts.values[1]}')

Date range: 2022-01-01 to 2024-06-30
Unique origins: 15
Unique destinations: 15
Transport modes: sea_freight 6426, air_freight 1574


## 2. Data Quality

In [4]:
print(f'Missing values: {shipments.isnull().sum().sum()}')
print(f'Duplicate shipment IDs: {shipments["shipment_id"].duplicated().sum()}')
print(f'Negative costs: {(shipments["cost_usd"] < 0).sum()}')
print(f'Negative delays: {(shipments["total_delay_days"] < 0).sum()}')

Missing values: 0
Duplicate shipment IDs: 0
Negative costs: 0
Negative delays: 0


## 3. Delay Overview

In [5]:
print('=== Delay Summary ===')
print(f'Total shipments: {len(shipments):,}')
delayed = shipments['is_delayed'].sum()
print(f'Delayed: {delayed:,} ({delayed/len(shipments):.1%})')
print(f'On-time: {len(shipments)-delayed:,} ({(len(shipments)-delayed)/len(shipments):.1%})')
print(f'Avg delay (all): {shipments["total_delay_days"].mean():.1f} days')
print(f'Avg delay (delayed only): {shipments[shipments["is_delayed"]]["total_delay_days"].mean():.1f} days')
print(f'Max delay: {shipments["total_delay_days"].max()} days')

=== Delay Summary ===
Total shipments: 8,000
Delayed: 4,337 (54.2%)
On-time: 3,663 (45.8%)
Avg delay (all): 1.9 days
Avg delay (delayed only): 3.6 days
Max delay: 11 days


In [6]:
# Delay rate over time
shipments['month'] = shipments['booking_date'].dt.to_period('M')
monthly = shipments.groupby('month').agg(
    total=('shipment_id', 'count'),
    delayed=('is_delayed', 'sum'),
    avg_delay=('total_delay_days', 'mean')
).reset_index()
monthly['delay_rate'] = monthly['delayed'] / monthly['total'] * 100
monthly['month'] = monthly['month'].dt.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(monthly['month'], monthly['total'], color='#3498db', width=20, alpha=0.6, label='Total')
axes[0].bar(monthly['month'], monthly['delayed'], color='#e74c3c', width=20, alpha=0.8, label='Delayed')
axes[0].set_title('Monthly Shipment Volume & Delays', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Shipments')
axes[0].legend()

axes[1].plot(monthly['month'], monthly['delay_rate'], 'o-', color='#e74c3c', linewidth=1.5)
axes[1].axhline(y=monthly['delay_rate'].mean(), color='gray', linestyle='--', alpha=0.7, label='Average')
axes[1].set_title('Monthly Delay Rate (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Delay Rate (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/monthly_delays.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Delay Causes

In [7]:
delayed_shipments = shipments[shipments['is_delayed'] & (shipments['delay_cause'] != '')]

cause_counts = delayed_shipments['delay_cause'].value_counts()
cause_avg_delay = delayed_shipments.groupby('delay_cause')['total_delay_days'].mean().reindex(cause_counts.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cause_counts.plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('Delay Causes — Frequency', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')

cause_avg_delay.sort_values().plot(kind='barh', ax=axes[1], color='#e67e22', edgecolor='white')
axes[1].set_title('Delay Causes — Avg Delay (days)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Days')

plt.tight_layout()
plt.savefig('../reports/figures/delay_causes.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Performance by Transport Mode

In [8]:
mode_stats = shipments.groupby('transport_mode').agg(
    total=('shipment_id', 'count'),
    delayed=('is_delayed', 'sum'),
    avg_delay_all=('total_delay_days', 'mean')
)
mode_stats['delay_rate'] = mode_stats['delayed'] / mode_stats['total'] * 100

mode_delayed = shipments[shipments['is_delayed']].groupby('transport_mode')['total_delay_days'].mean()

print('Delay rate by mode:')
for mode in mode_stats.index:
    print(f'  {mode}: {mode_stats.loc[mode, "delay_rate"]:.1f}%')

print('\nAvg delay by mode (delayed only):')
for mode in mode_delayed.index:
    print(f'  {mode}: {mode_delayed[mode]:.1f} days')

Delay rate by mode:
  sea_freight: 55.8%
  air_freight: 47.6%

Avg delay by mode (delayed only):
  sea_freight: 3.7 days
  air_freight: 3.2 days


## 6. Performance by Carrier

In [9]:
carrier_stats = shipments.groupby('carrier').agg(
    shipments=('shipment_id', 'count'),
    delay_rate=('is_delayed', 'mean'),
    avg_delay=('total_delay_days', 'mean'),
    avg_cost=('cost_usd', 'mean')
).round(2)
carrier_stats['delay_rate'] = (carrier_stats['delay_rate'] * 100).round(1)
carrier_stats = carrier_stats.sort_values('delay_rate', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if r > 55 else '#e67e22' if r > 50 else '#27ae60' for r in carrier_stats['delay_rate']]
ax.barh(carrier_stats.index, carrier_stats['delay_rate'], color=colors, edgecolor='white')
ax.axvline(x=carrier_stats['delay_rate'].mean(), color='gray', linestyle='--', alpha=0.7, label='Average')
ax.set_title('Delay Rate by Carrier (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Delay Rate (%)')
ax.legend()

for i, v in enumerate(carrier_stats['delay_rate']):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/carrier_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Seasonal Patterns

In [10]:
shipments['booking_month'] = shipments['booking_date'].dt.month
monthly_delay = shipments.groupby('booking_month')['is_delayed'].mean() * 100

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if m in [6,7,8,9] else '#3498db' for m in range(1, 13)]
ax.bar(range(1, 13), monthly_delay.values, color=colors, edgecolor='white')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.axhline(y=monthly_delay.mean(), color='gray', linestyle='--', alpha=0.7, label='Average')
ax.set_title('Delay Rate by Month — Monsoon Effect (Jun-Sep highlighted)', fontsize=12, fontweight='bold')
ax.set_ylabel('Delay Rate (%)')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/seasonal_delays.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Top Delayed Routes

In [11]:
shipments['route'] = shipments['origin_port'] + ' \u2192 ' + shipments['destination_port']

route_stats = shipments.groupby('route').agg(
    shipments=('shipment_id', 'count'),
    delay_rate=('is_delayed', 'mean'),
    avg_delay=('total_delay_days', 'mean')
).round(2)
route_stats['delay_rate'] = (route_stats['delay_rate'] * 100).round(1)

top_routes = route_stats[route_stats['shipments'] >= 30].sort_values('delay_rate', ascending=False).head(10)

print('Top 10 routes by delay rate (min 30 shipments):')
print(top_routes.to_string())

Top 10 routes by delay rate (min 30 shipments):


## 9. Cost Analysis

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost by mode
shipments.groupby('transport_mode')['cost_usd'].mean().plot(
    kind='bar', ax=axes[0], color=['#3498db', '#e67e22'], edgecolor='white'
)
axes[0].set_title('Avg Shipment Cost by Mode (USD)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', rotation=0)

# Cost vs delay
axes[1].scatter(shipments['total_delay_days'], shipments['cost_usd'], alpha=0.15, s=8, color='#2c3e50')
axes[1].set_title('Cost vs Delay', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delay (days)')
axes[1].set_ylabel('Cost (USD)')

plt.tight_layout()
plt.savefig('../reports/figures/cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Findings

1. **54.2% of shipments are delayed** \u2014 over half of all deliveries miss scheduled arrival
2. **Average delay: 3.6 days** (when delayed), max 11 days
3. **Top delay causes**: port congestion, customs clearance, weather
4. **Sea freight delays higher** (55.8%) than air freight (47.6%)
5. **Monsoon season (Jun-Sep)** shows elevated delay rates for SEA routes
6. **COVID hangover**: 2022 H1 had notably higher port congestion
7. **Carrier variance**: delay rates range from ~48% to ~58% across carriers

**Next:** KPI dashboard for COO, detailed route optimization recommendations.